# Log Analysis & Anomaly Detection

## What This Is
System and application logs contain rich security signals buried in high-volume, semi-structured text. Log analysis for security involves:

1. **Parsing**: Extract structure from free-form log lines (Drain algorithm, regex templates)
2. **Template mining**: Identify recurring log patterns and their parameters
3. **Anomaly scoring**: Detect deviations from normal log sequences or parameter values

## Drain (2017)
Drain is the state-of-the-art online log parser. It uses a prefix tree (trie) where:
- The root branches on log length
- The first node at each length branches on the first token
- Leaf nodes store log templates with wildcard placeholders for variable tokens

Drain processes millions of log lines per second with O(k) per-line complexity where k is the log depth parameter.

In [ ]:
import numpy as np
import re
from typing import List, Dict, Tuple, Optional
from collections import defaultdict

np.random.seed(42)

# Simplified Drain-style log parser

class DrainNode:
    def __init__(self):
        self.children: Dict[str, 'DrainNode'] = {}
        self.templates: List[List[str]] = []
        self.template_ids: List[int] = []

class SimpleDrain:
    """
    Simplified Drain log parser.
    Builds a prefix tree indexed by (log_length, first_token).
    """
    
    WILDCARD = '<*>'
    
    def __init__(self, depth: int = 4, similarity_threshold: float = 0.5):
        self.depth = depth
        self.sim_threshold = similarity_threshold
        self.root: Dict = {}  # {length: {first_token: [templates]}}
        self.templates: Dict[int, List[str]] = {}  # template_id -> template tokens
        self.template_count: Dict[int, int] = {}
        self.next_id = 0
    
    def tokenize(self, line: str) -> List[str]:
        # Replace numbers/IPs/paths with wildcards before tokenizing
        line = re.sub(r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', '<IP>', line)
        line = re.sub(r'\b\d+\b', '<NUM>', line)
        return line.strip().split()
    
    def _similarity(self, tokens: List[str], template: List[str]) -> float:
        if len(tokens) != len(template):
            return 0.0
        matches = sum(1 for t, p in zip(tokens, template) if t == p or p == self.WILDCARD)
        return matches / len(tokens)
    
    def _update_template(self, tokens: List[str], template: List[str]) -> List[str]:
        """Merge tokens into template, replacing non-matching positions with wildcard."""
        return [t if t == p else self.WILDCARD for t, p in zip(tokens, template)]
    
    def parse(self, line: str) -> Tuple[int, str]:
        """Return (template_id, template_string)."""
        tokens = self.tokenize(line)
        n = len(tokens)
        first = tokens[0] if tokens else ''
        
        # Get candidate templates for this (length, first_token)
        key = (n, first)
        if key not in self.root:
            self.root[key] = []
        
        candidates = self.root[key]
        
        # Find best matching template
        best_sim, best_idx = 0.0, -1
        for i, tid in enumerate(candidates):
            sim = self._similarity(tokens, self.templates[tid])
            if sim > best_sim:
                best_sim = sim
                best_idx = i
        
        if best_sim >= self.sim_threshold:
            # Update existing template
            tid = candidates[best_idx]
            self.templates[tid] = self._update_template(tokens, self.templates[tid])
            self.template_count[tid] += 1
            return tid, ' '.join(self.templates[tid])
        else:
            # New template
            tid = self.next_id
            self.next_id += 1
            self.templates[tid] = tokens[:]
            self.template_count[tid] = 1
            candidates.append(tid)
            return tid, ' '.join(tokens)

# Test with sample log lines
sample_logs = [
    'sshd[1234]: Failed password for root from 192.168.1.100 port 54321',
    'sshd[1235]: Failed password for admin from 192.168.1.101 port 54322',
    'sshd[1236]: Failed password for user from 10.0.0.5 port 12345',
    'sshd[1237]: Accepted password for john from 192.168.1.50 port 44444',
    'sshd[1238]: Accepted publickey for john from 192.168.1.50 port 44445',
    'kernel: TCP: request_sock_TCP: Possible SYN flooding on port 80',
    'kernel: TCP: request_sock_TCP: Possible SYN flooding on port 443',
    'apache: GET /admin/config.php HTTP 200 from 10.0.0.9',
    'apache: GET /wp-admin/login HTTP 404 from 10.0.0.9',
]

parser = SimpleDrain()
print('Drain Log Parsing Results:')
print('-' * 70)
for log in sample_logs:
    tid, template = parser.parse(log)
    print(f'  T{tid:02d}: {template[:65]}')

print(f'\nUnique templates discovered: {len(parser.templates)}')


In [ ]:
# Anomaly detection using template frequency
# Unusual templates = anomalous log events

def generate_normal_logs(n: int) -> List[str]:
    """Generate normal operational log entries."""
    templates = [
        'sshd[{pid}]: Accepted password for {user} from {ip} port {port}',
        'sshd[{pid}]: session opened for user {user} by (uid=0)',
        'sshd[{pid}]: session closed for user {user}',
        'kernel: EXT4-fs (sda1): recovery complete',
        'systemd: Started {service}.service',
        'cron[{pid}]: ({user}) CMD ({cmd})',
    ]
    logs = []
    for _ in range(n):
        t = np.random.choice(templates)
        log = t.format(
            pid=np.random.randint(1000, 9999),
            user=np.random.choice(['john', 'alice', 'bob', 'service']),
            ip=f'192.168.{np.random.randint(0,3)}.{np.random.randint(1,50)}',
            port=np.random.randint(10000, 65535),
            service=np.random.choice(['nginx', 'mysql', 'redis']),
            cmd=np.random.choice(['backup.sh', 'cleanup.py', 'monitor.sh'])
        )
        logs.append(log)
    return logs

def generate_attack_logs(n: int) -> List[str]:
    """Generate attack log entries (brute force + exploitation)."""
    templates = [
        'sshd[{pid}]: Failed password for root from {ip} port {port}',
        'sshd[{pid}]: Failed password for admin from {ip} port {port}',
        'apache: GET /etc/passwd HTTP 200 from {ip}',
        'apache: GET /admin/shell.php HTTP 200 from {ip}',
        'kernel: TCP: SYN flooding on port {port} from {ip}',
    ]
    logs = []
    for _ in range(n):
        t = np.random.choice(templates)
        log = t.format(
            pid=np.random.randint(1000, 9999),
            ip=f'10.evil.{np.random.randint(0,255)}.{np.random.randint(1,255)}',
            port=np.random.randint(1, 1024)
        )
        logs.append(log)
    return logs

# Parse and score
normal_logs = generate_normal_logs(500)
attack_logs = generate_attack_logs(50)

parser2 = SimpleDrain()
# Build baseline on normal logs
for log in normal_logs:
    parser2.parse(log)

# Record baseline template frequencies
baseline_counts = dict(parser2.template_count)
total_baseline = sum(baseline_counts.values())

# Score attack logs: templates not seen in baseline = anomalous
print('Attack Log Anomaly Scoring:')
anomalies = 0
for log in attack_logs[:10]:
    tid, template = parser2.parse(log)
    baseline_freq = baseline_counts.get(tid, 0) / total_baseline
    is_anomaly = baseline_freq < 0.001
    if is_anomaly:
        anomalies += 1
    print(f'  {'[ANOMALY]' if is_anomaly else '[NORMAL ]'} freq={baseline_freq:.4f} | {template[:60]}')

print(f'\nAnomaly detection rate: {anomalies}/10 attack logs flagged')
